# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SammyFarrelZebua/flyrank-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


---

## Your turn — Exploratory Data Analysis (EDA)

*EDA on the full warehouse using DuckDB SQL aggregations — no data downloaded, only summaries returned to pandas.*

In [ ]:
# 1. Warehouse Schema & Column Overview
from IPython.display import display

print("=== WAREHOUSE SCHEMA & COLUMN LIST ===\n")
for name, src in TABLES.items():
    schema = con.sql(f"DESCRIBE SELECT * FROM {src}").df()
    print(f"--- Table: {name} ({len(schema)} columns) ---")
    display(schema[['column_name', 'column_type']])
    print("-" * 50)

In [ ]:
# 2. Missing Value Mapping & Availability Flags

# A. dim_content Missingness
null_dim_content = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(word_count) AS null_word_count,
        ROUND((COUNT(*) - COUNT(word_count)) * 100.0 / COUNT(*), 2) AS null_pct_word_count,
        COUNT(*) - COUNT(search_volume) AS null_search_volume,
        ROUND((COUNT(*) - COUNT(search_volume)) * 100.0 / COUNT(*), 2) AS null_pct_search_volume
    FROM {TABLES['dim_content']}
""").df()

print("[A] dim_content Missingness:")
display(null_dim_content.T)

# B. fact_daily_sample Missingness & Availability Flags
null_fact_daily = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows_in_sample,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_gsc_impressions,
        SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS zero_gsc_position_no_rank_data,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_not_available_rows,
        ROUND(SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ga4_not_available_pct
    FROM {TABLES['fact_daily_sample']}
""").df()

print("\n[B] fact_daily_sample Missingness & Availability:")
display(null_fact_daily.T)

In [ ]:
# 3. Descriptive Statistics (Mean, Median, Std, Quantiles)

# A. Daily Performance Metrics (GSC & GA4)
stats_fact_daily = con.sql(f"""
    SELECT 
        'gsc_impressions' AS metric,
        AVG(gsc_impressions) AS mean,
        MEDIAN(gsc_impressions) AS median,
        STDDEV_SAMP(gsc_impressions) AS std,
        MIN(gsc_impressions) AS min,
        QUANTILE_CONT(gsc_impressions, 0.25) AS q25,
        QUANTILE_CONT(gsc_impressions, 0.75) AS q75,
        MAX(gsc_impressions) AS max
    FROM {TABLES['fact_daily_sample']}
    UNION ALL
    SELECT 
        'gsc_clicks' AS metric,
        AVG(gsc_clicks), MEDIAN(gsc_clicks), STDDEV_SAMP(gsc_clicks),
        MIN(gsc_clicks), QUANTILE_CONT(gsc_clicks, 0.25), QUANTILE_CONT(gsc_clicks, 0.75), MAX(gsc_clicks)
    FROM {TABLES['fact_daily_sample']}
    UNION ALL
    SELECT 
        'gsc_avg_position (where > 0)' AS metric,
        AVG(gsc_avg_position), MEDIAN(gsc_avg_position), STDDEV_SAMP(gsc_avg_position),
        MIN(gsc_avg_position), QUANTILE_CONT(gsc_avg_position, 0.25), QUANTILE_CONT(gsc_avg_position, 0.75), MAX(gsc_avg_position)
    FROM {TABLES['fact_daily_sample']}
    WHERE gsc_avg_position > 0
""").df()

print("[A] Daily Fact Performance Statistics:")
display(stats_fact_daily)

# B. Content Metadata Metrics
stats_dim_content = con.sql(f"""
    SELECT 
        'word_count' AS metric,
        AVG(word_count) AS mean,
        MEDIAN(word_count) AS median,
        STDDEV_SAMP(word_count) AS std,
        MIN(word_count) AS min,
        QUANTILE_CONT(word_count, 0.25) AS q25,
        QUANTILE_CONT(word_count, 0.75) AS q75,
        MAX(word_count) AS max
    FROM {TABLES['dim_content']}
    WHERE word_count IS NOT NULL
    UNION ALL
    SELECT 
        'search_volume' AS metric,
        AVG(search_volume), MEDIAN(search_volume), STDDEV_SAMP(search_volume),
        MIN(search_volume), QUANTILE_CONT(search_volume, 0.25), QUANTILE_CONT(search_volume, 0.75), MAX(search_volume)
    FROM {TABLES['dim_content']}
    WHERE search_volume IS NOT NULL
""").df()

print("\n[B] Content Metadata Statistics:")
display(stats_dim_content)

In [ ]:
# 4. Cardinality & Categorical Breakdown

cardinality = con.sql(f"""
    SELECT 
        (SELECT COUNT(DISTINCT client_hash_id) FROM {TABLES['dim_clients']}) AS unique_clients,
        (SELECT COUNT(DISTINCT content_hash_id) FROM {TABLES['dim_content']}) AS unique_content_items,
        (SELECT COUNT(DISTINCT content_type) FROM {TABLES['dim_content']}) AS unique_content_types
""").df()

print("=== CARDINALITY SUMMARY ===")
display(cardinality)

print("\n=== CONTENT TYPE BREAKDOWN ===")
content_types = con.sql(f"""
    SELECT content_type, COUNT(*) AS count, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
    FROM {TABLES['dim_content']}
    GROUP BY content_type
    ORDER BY count DESC
""").df()
display(content_types)